[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reinhart-group/generative-copolymer-workshop/blob/main/day2/05_mapping.ipynb)

# Day 2 — Async: Mapping to the Master Dataset

**Objectives:**
- Pool your feature vectors with the provided master dataset
- Run UMAP to locate your simulated sequence in the global structure space
- Prepare sequence data ($X$) and UMAP coordinates ($Z$) for regression

**The big picture.** Day 1 gave you CNN feature vectors for a handful of sequences *you* simulated. Notebook 04 showed you how to compress those into a 2-D map with PCA and UMAP — but a map built only from your own sequences is tiny. You can't tell whether your structure is common or rare, or which morphological region it sits in.

This notebook solves that by merging your feature vectors with a **master dataset** of ~16k pre-computed reference sequences, fitting a *single* UMAP on the combined matrix, and placing your sequence on the global morphology map. At the end you will have a `(Z0, Z1)` coordinate for every sequence in the library, ready to hand to the regression model in notebook 06.

In [ ]:
# If running on Colab, uncomment and run this cell first, then restart the runtime:
# !pip install "umap-learn>=0.5.6,<0.6" "scikit-learn>=1.3,<2" tqdm

# To verify installed versions after installing:
# import umap, sklearn; print(umap.__version__, sklearn.__version__)

**The toolkit.** We will be using:

- `numpy` — numerical arrays
- `pandas` — tabular data
- `matplotlib` — plots
- `scikit-learn` (`StandardScaler`) — feature standardization, available if needed
- `umap-learn` (`umap`) — non-linear dimensionality reduction
- `os`, `ast`, `glob` — file discovery and string parsing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os, ast, glob
from sklearn.preprocessing import StandardScaler
import umap

## Helper functions

The next cell defines utility functions used throughout the notebook. `pool_features` averages feature vectors across multiple views and bead-pair combinations into one representative vector. `load_feature_csv` loads a pre-pooled feature CSV and converts the stored string arrays back into numpy arrays.

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def beads_match(stored, target):
    """Return True if a stored bead combo (string or list) matches target, in any order."""
    try:
        stored_list = ast.literal_eval(stored) if isinstance(stored, str) else stored
        return sorted(stored_list) == sorted(target)
    except Exception:
        return False


def pool_features(features, pool_type='mean'):
    """Combine a list of feature vectors into one by pooling ('mean' or 'max')."""
    features = np.array(features)
    if pool_type == 'mean':
        return np.mean(features, axis=0)
    elif pool_type == 'max':
        return np.max(features, axis=0)
    else:
        raise ValueError(f"pool_type must be 'mean' or 'max', got '{pool_type}'.")


def get_features_all_combos(df, bead_combinations, views, pool_type='mean'):
    """
    Pool CNN features across all bead-pair combinations and viewing angles.
    Returns a DataFrame with one row per sequence (columns: Sequence, Feature).
    """
    bead_mask = df['Beads'].apply(
        lambda b: any(beads_match(b, combo) for combo in bead_combinations)
    )
    view_mask = df['View'].isin(views)
    filtered  = df[bead_mask & view_mask].copy()
    if len(filtered) == 0:
        raise ValueError('No matching rows — check bead names and view labels.')
    n_combos = filtered['Beads'].nunique()
    n_views  = filtered['View'].nunique()
    print(f'  Found {n_combos} bead combos and {n_views} views '
          f'across {filtered["Sequence"].nunique()} sequences')
    records = []
    for seq, group in filtered.groupby('Sequence'):
        records.append({'Sequence': seq, 'Feature': pool_features(list(group['Feature']), pool_type)})
    return pd.DataFrame(records)


def load_feature_csv(path):
    """Load a feature CSV (columns: Sequence, Feature) and parse the Feature column."""
    df = pd.read_csv(path, dtype={'Sequence': str})
    df['Feature'] = df['Feature'].apply(
        lambda x: np.array(ast.literal_eval(x), dtype=np.float32)
    )
    return df

## 1. Loading the Master Dataset

The **master dataset** contains pre-computed, pooled CNN feature vectors for every sequence
in the reference library (~16k binary copolymer sequences). It was prepared by the instructors
using the same pipeline you ran in Day 1 and is available in the shared Google Drive folder.

Your goal is to combine your own sequence(s) with this library so the UMAP in Section 2
places your data in a globally meaningful context — not just relative to itself.

We load data in four steps — set options, load the master dataset, load your own feature CSVs from Day 1, and pool everything down to one vector per sequence. The result is two feature matrices (`master_features` and `your_features`) with identical column layouts, ready to be stacked and passed to UMAP.

**Step 1a — Set your options.** These settings must match what you used in Day 1. The model name, KDE sigma, bead combinations, and viewing angles determine which files get loaded and how feature vectors are named.

In [ ]:
# --- OPTIONS (must match your Day 1 settings) ---
MODEL_NAME = 'efficientnet_b0'
KDE_SIGMA  = 2.0
MODEL_IMG_SIZE = {
    'resnet50': 224, 'efficientnet_b0': 224,
    'efficientnet_b1': 240, 'efficientnet_b2': 260,
}
IMG_SIZE       = MODEL_IMG_SIZE[MODEL_NAME]
pool_type      = 'mean'
name_extension = f'_cv2_kde_sigma_{KDE_SIGMA}_size_{IMG_SIZE}.png'
df_file        = f'img_feats_{MODEL_NAME}{name_extension[:-4]}.csv'

bead_combinations = [['SN3r', 'C4'], ['SN3r', 'TP1'], ['C4', 'TP1']]
views             = ['xy', 'xz', 'yz']

**Step 1b — Set paths and load the master dataset.** Point `your_folder_path` at the folder of per-structure results from Day 1. Set `master_features_path` to the path of the pre-pooled master CSV downloaded from Google Drive. The master file has the same schema as your own feature files — one row per sequence with a pooled 1280-dimensional feature vector.

In [ ]:
# --- PATHS (update these) ---
your_folder_path     = '/noether/s1/kac6810/a_compare_embeddings/high_res_imgs/traj_data'
master_features_path = 'master_pooled_features.csv'   # download from Google Drive and set path

# Load the master feature dataset
master_df       = load_feature_csv(master_features_path)
master_features = np.vstack(master_df['Feature'].values)
master_seqs     = master_df['Sequence'].values
print(f'Master dataset: {master_features.shape[0]} sequences × {master_features.shape[1]} features')

**Step 1c — Load your feature files from Day 1.** Each sequence folder contains a CSV with one row per (bead pair × view angle) combination. We discover all available CSVs, load them, and stack them into one combined table.

In [ ]:
# Discover sequence folders
seqs = [s for s in os.listdir(your_folder_path)
        if os.path.isdir(os.path.join(your_folder_path, s))]

csv_files = [os.path.join(your_folder_path, seq, df_file) for seq in seqs]
existing  = [f for f in csv_files if os.path.exists(f)]
missing   = len(csv_files) - len(existing)
print(f'Loading {len(existing)}/{len(csv_files)} feature CSVs'
      + (f'  ({missing} missing)' if missing else ''))

dfs = []
for csv_path in existing:
    df = pd.read_csv(csv_path, dtype={'Sequence': str})
    df['Feature'] = df['Feature'].apply(
        lambda x: np.array(ast.literal_eval(x), dtype=np.float32)
    )
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(combined_df)}  '
      f'(each row = one bead combo × one view × one sequence)')

**Step 1d — Pool into one vector per sequence.** Average all views and bead-pair feature vectors for each of your sequences into a single representative fingerprint, matching the format of the master dataset.

In [ ]:
# Pool across all bead combos and views → one feature vector per sequence
your_pooled   = get_features_all_combos(combined_df, bead_combinations, views, pool_type)
your_features = np.vstack(your_pooled['Feature'].values)
your_seqs     = your_pooled['Sequence'].values

print(f'\nYour sequences: {len(your_seqs)}  '
      f'(feature vector length: {your_features.shape[1]})')

▶ **What you should see:** two printouts — the master dataset dimensions (something like `16 500 sequences × 1280 features`) and your own sequence count. If the master dataset fails to load, check `master_features_path`. If your sequence count is 0, the Day-1 CSVs were not found at `your_folder_path`.

## 2. Pooling and Embedding

Now we combine the master features with your own feature vectors and run UMAP on the
full combined matrix.

Because UMAP is fit on all sequences at once, your sequences are placed in the **same
coordinate system** as the master library — you can directly compare where you land
relative to known morphological regions (strings, vesicles, micelles, etc.).

**Why run UMAP on the combined matrix?** If we ran UMAP separately on the master library and on your sequences, the two results would be in entirely different coordinate systems — like two city maps printed at different scales and rotations. By stacking all feature vectors into *one* matrix and fitting a single UMAP, every sequence lands in the same global coordinate system. The master library anchors the map; your sequences are placed relative to it.

> **A note on scale:** with ~16k master sequences and potentially only a handful of your own, your sequences won't noticeably distort the UMAP layout — they will simply be embedded into the map the master library defines. This is exactly what we want: an unbiased global view.

In [ ]:
# Stack master + your features into one matrix
# We keep track of the boundary so we can separate them after UMAP
n_master      = len(master_seqs)
all_features  = np.vstack([master_features, your_features])
all_sequences = np.concatenate([master_seqs, your_seqs])

print(f'Combined matrix: {all_features.shape[0]} sequences × {all_features.shape[1]} features')
print(f'  ({n_master} master  +  {len(your_seqs)} yours)')

# --- RUN UMAP ---
reducer     = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
umap_coords = reducer.fit_transform(all_features)

master_coords = umap_coords[:n_master]
your_coords   = umap_coords[n_master:]

print(f'\nUMAP done. Coordinates shape: {umap_coords.shape}')

▶ **What you should see:** a line like `UMAP done. Coordinates shape: (16xxx, 2)`. The shape confirms every sequence — master and yours — received a 2-D coordinate. UMAP typically takes 20–60 seconds on ~16k sequences; if it seems to hang, check available memory.

## 3. Where Did Your Sequence Land?

Plot the global UMAP — all master sequences as small blue dots, your sequence(s) highlighted in red and labelled with their binary strings.

This is the payoff: you can now see exactly where your polymer chemistry falls on the global morphology map. Think about:
- **Location** — which broad morphological region (strings, vesicles, micelles) is your sequence closest to?
- **Density** — is your point inside a dense cluster (common morphology) or in a sparse area (potentially novel)?
- **Expectation** — if you designed the sequence with a target morphology in mind, does its map position match that expectation?
- **Chemistry** — how does the position relate to the block ratio or hydrophilic/hydrophobic balance of your sequence?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(master_coords[:, 0], master_coords[:, 1],
           s=4, alpha=0.2, color='steelblue', label=f'Master ({n_master} seqs)')
ax.scatter(your_coords[:, 0], your_coords[:, 1],
           s=120, color='red', zorder=5, edgecolors='black', linewidths=0.5,
           label=f'Your sequence(s) ({len(your_seqs)})')

# Label each of your sequences with its binary string
for i, (x, y) in enumerate(your_coords):
    ax.annotate(your_seqs[i], (x, y),
                textcoords='offset points', xytext=(6, 4),
                fontsize=7, color='darkred')

ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('Global Structure Map — Your Sequence in Context')
ax.legend(markerscale=2, loc='best')
plt.tight_layout()
plt.show()

▶ **What you should see:** a blue cloud of master library points with red markers on top for your sequence(s), each labelled with its binary string. If no red points appear, `your_seqs` is likely empty — re-run Steps 1c–1d. If a red point sits deep inside a dense blue cluster, your sequence shares its morphology with many library members; a point near the fringes may represent something less common.

## 4. Preparing Data for Regression

To train the RNN in notebook 06, we need:
- **X** — each sequence encoded as a length-30 array of integers (0 or 1)
- **Z** — the corresponding 2-D UMAP coordinates

We use the *full combined dataset* (master + yours) so the RNN learns from all available
sequence–structure pairs. We also save the embedding to a CSV so notebook 06 can load it
directly without re-running UMAP.

**How we encode sequences.** Each binary copolymer is a string of `0`s and `1`s, e.g. `'011100101100...'`. To feed it to a neural network we convert each character to an integer, yielding a fixed-length vector of `sequence_length` values. No tokeniser or embedding table is needed — the two-character alphabet is trivially represented as binary integers.

**What goes into the saved file.** `umap_embedding_for_regression.csv` has three columns: `Sequence`, `Z0`, and `Z1`. Notebook 06 loads this file and trains an RNN to predict `(Z0, Z1)` from the sequence string. That learned mapping is the **forward model** for inverse design: given any sequence, predict where it lands on the global structure map.

In [ ]:
# X: encode each sequence string as a list of 0/1 integers
# e.g.  '011100...' → [0, 1, 1, 1, 0, 0, ...]
X = np.array([[int(c) for c in seq] for seq in all_sequences], dtype=np.float32)

# Z: UMAP 2-D coordinates
Z = umap_coords.astype(np.float32)

print(f'X shape: {X.shape}  (n_sequences × sequence_length)')
print(f'Z shape: {Z.shape}  (n_sequences × 2 UMAP dims)')

# Quick sanity check — show a few rows
for i in range(3):
    print(f'  seq={all_sequences[i]}  Z=({Z[i,0]:.3f}, {Z[i,1]:.3f})')

# Save so notebook 06 can load this without re-running UMAP
embedding_df = pd.DataFrame({
    'Sequence': all_sequences,
    'Z0': Z[:, 0],
    'Z1': Z[:, 1],
})
embedding_df.to_csv('umap_embedding_for_regression.csv', index=False)
print(f'\nSaved to umap_embedding_for_regression.csv  ({len(embedding_df)} rows)')

▶ **What you should see:** shapes like `X shape: (16xxx, 30)` and `Z shape: (16xxx, 2)` — confirming sequence length and embedding dimensionality — followed by three sample rows. The file `umap_embedding_for_regression.csv` should now appear in your working directory. Notebook 06 expects this exact filename.